<a href="https://colab.research.google.com/github/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/blob/main/Notebooks/M04_02_Regresi%C3%B3n_lineal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
  <img src="https://github.com/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/raw/main/Static/Imgs/UdeC_color_horizontal.jpg" width="500">
</p>

<p align="center"><b style="font-size:28px;">Facultad de Ingeniería Agrícola</b></p>
<p align="center"><b style="font-size:28px;">Introducción a la Ciencia de Datos</b></p>
<hr>
<p align="center"><b style="font-size:28px;">Contacto</b></p>
<p align="center">
  paulmunoz@udec.cl<br>
  https://paulmunoz.com
</p>

## Módulo 4: Análisis e interpretación de datos

### Notebook 2: Regresión lineal

En la actividad anterior estudiamos cómo identificar y cuantificar relaciones entre variables mediante los coeficientes de correlación de Pearson y Spearman. También aprendimos a interpretar la significancia estadística y a distinguir entre correlación y causalidad.

Sin embargo, conocer la fuerza de una relación no nos permite estimar directamente el valor de una variable a partir de otra.

Por ejemplo, utilizando los datos de **MeteoChillán**, identificamos una correlación negativa fuerte entre la temperatura del aire y la humedad relativa.

A partir de este resultado surgen nuevas preguntas:

- ¿Podemos estimar la humedad relativa a partir de la temperatura?
- ¿Cuánto cambia, en promedio, la humedad relativa cuando aumenta la temperatura en 1 °C?
- ¿Cómo podemos construir una ecuación que represente esta relación?
- ¿Qué tan precisas son las estimaciones obtenidas?
- ¿Cómo podemos evaluar si un modelo representa adecuadamente los datos?

Para responder estas preguntas utilizaremos la **regresión lineal**, uno de los métodos estadísticos más utilizados para analizar relaciones entre variables cuantitativas y construir modelos predictivos.

Trabajaremos con los mismos datos meteorológicos de MeteoChillán utilizados en el notebook anterior. Estos datos ya fueron sometidos a un control básico de calidad, agregados a resolución horaria y exportados a un archivo CSV.

En este notebook aprenderemos a construir, evaluar e interpretar un modelo de regresión lineal simple utilizando Python.

Al finalizar este notebook podrás:

- Diferenciar entre correlación y regresión lineal.
- Comprender la ecuación de regresión lineal y el significado de sus coeficientes.
- Explicar cómo funciona el método de mínimos cuadrados.
- Ajustar un modelo de regresión lineal utilizando datos ambientales.
- Obtener estimaciones e interpretar los residuos del modelo.
- Evaluar el desempeño mediante R², MAE y RMSE.
- Interpretar la significancia estadística y los intervalos de confianza de los coeficientes.
- Reconocer las principales limitaciones de la regresión lineal.

## 1. Importación de librerías

Para desarrollar esta actividad utilizaremos las siguientes librerías de Python:

- **Pandas:** importar y manipular los datos meteorológicos.
- **NumPy:** realizar operaciones numéricas.
- **Matplotlib y Seaborn:** visualizar las observaciones, las relaciones entre variables y los resultados de los modelos.
- **SciPy:** realizar análisis estadísticos.
- **Statsmodels:** ajustar modelos de regresión lineal y obtener información estadística sobre sus coeficientes.
- **Scikit-learn:** construir modelos de regresión y evaluar su desempeño mediante diferentes métricas.

Estas librerías nos permitirán combinar el análisis estadístico con herramientas de modelación y evaluación predictiva.

In [ ]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Análisis estadístico
from scipy import stats
import statsmodels.api as sm

# Regresión lineal y evaluación de modelos
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configuración de gráficos
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 120

## 2. Carga y exploración de datos

En esta actividad utilizaremos el conjunto de datos meteorológicos de **MeteoChillán** preparado en el notebook anterior (`M04_01_Correlacion_causalidad`).

Durante esa sesión realizamos el control básico de calidad, la selección de variables y la agregación de los registros a resolución horaria. Finalmente, exportamos los datos procesados a un archivo CSV.

En este notebook **no repetiremos la limpieza de datos**. Trabajaremos directamente con el archivo exportado.

Las variables disponibles son:

| Variable | Descripción | Unidad |
|---|---|---|
| Temperatura | Temperatura del aire | °C |
| Humedad | Humedad relativa del aire | % |
| Radiación | Radiación solar | W/m² |
| Viento | Velocidad del viento | m/s |
| Presión | Presión atmosférica | hPa |

### 2.1. Importación del archivo CSV

Para comenzar:

1. Localice el archivo CSV exportado durante la actividad anterior.
2. Ejecute la siguiente celda y seleccione el archivo desde su computador.
3. Compruebe que los datos se hayan importado correctamente.

Utilizaremos **Pandas** para leer el archivo y establecer la fecha y hora como índice temporal del DataFrame.

In [ ]:
from google.colab import files

# Seleccionar el archivo CSV exportado en el notebook anterior
uploaded = files.upload()

# Obtener el nombre del archivo seleccionado
nombre_archivo = next(iter(uploaded))

# Importar los datos
df_final = pd.read_csv(
    nombre_archivo,
    index_col=0,
    parse_dates=True
)

# Ordenar cronológicamente
df_final = df_final.sort_index()

# Mostrar información general
print(f"Archivo: {nombre_archivo}")
print(f"Dimensiones: {df_final.shape[0]} registros y {df_final.shape[1]} variables")
print(f"Periodo: {df_final.index.min()} a {df_final.index.max()}")

# Mostrar las primeras observaciones
display(df_final.head())

### 2.2. Verificación de los datos importados

Antes de construir un modelo de regresión, debemos comprobar que los datos se hayan importado correctamente.

En particular, verificaremos:

1. **Tipos de datos:** las variables meteorológicas deben ser numéricas.
2. **Valores faltantes:** identificaremos cuántas observaciones no están disponibles para cada variable.
3. **Estadísticos descriptivos:** revisaremos los valores mínimos, máximos, medias y desviaciones estándar.

Como los datos ya fueron procesados en la actividad anterior, esta revisión no busca repetir la limpieza, sino confirmar que el archivo está preparado para el análisis.

**Importante:** para ajustar un modelo de regresión lineal necesitamos observaciones válidas tanto de la variable predictora (X) como de la variable que queremos estimar (Y). Por ello, posteriormente seleccionaremos únicamente los pares de observaciones disponibles.

In [ ]:
# Tipos de datos
print("Tipos de datos:")
print(df_final.dtypes)

# Valores faltantes
print("\nValores faltantes por variable:")
print(df_final.isna().sum())

# Estadísticos descriptivos
print("\nEstadísticos descriptivos:")
display(df_final.describe().round(2))

## 3. Fundamentos de la regresión lineal

### 3.1. ¿Qué es la regresión lineal?

La **regresión lineal** es un método estadístico que permite describir y cuantificar la relación entre una variable que queremos explicar o estimar y una o más variables predictoras.

En la **regresión lineal simple** utilizamos dos variables:

- **Variable independiente o predictora (X):** variable que utilizamos para explicar o estimar otra variable.
- **Variable dependiente o respuesta (Y):** variable cuyo valor queremos estimar.

Por ejemplo, con los datos de MeteoChillán podemos estudiar cómo cambia la **humedad relativa (Y)** en función de la **temperatura del aire (X)**.

### ¿En qué se diferencia de la correlación?

En la actividad anterior utilizamos la correlación para cuantificar la fuerza y dirección de las relaciones entre variables.

La regresión lineal permite avanzar un paso más: **construir una ecuación matemática que describa esa relación y permita realizar estimaciones**.

| Característica | Correlación | Regresión lineal |
|---|---|---|
| Objetivo | Medir la fuerza y dirección de una asociación | Modelar la relación entre variables |
| Variables | No distingue entre X e Y | Distingue entre predictora (X) y respuesta (Y) |
| Resultado | Coeficiente de correlación (r) | Ecuación de regresión |
| Unidades | Coeficiente sin unidades | Coeficientes con unidades |
| Estimaciones | No permite estimar directamente Y | Permite estimar Y a partir de X |

**Ejemplo:**

En el notebook anterior encontramos una correlación negativa fuerte entre temperatura y humedad relativa (r ≈ -0.87).

Este resultado indica que las temperaturas más altas tienden a estar asociadas con humedades relativas más bajas.

Sin embargo, la correlación no responde directamente preguntas como:

- ¿Cuántos puntos porcentuales disminuye, en promedio, la humedad relativa cuando la temperatura aumenta 1 °C?
- ¿Qué humedad relativa estimaría el modelo para una temperatura de 20 °C?

Para responder estas preguntas necesitamos ajustar un modelo de regresión.

**Importante:** al igual que la correlación, la regresión lineal por sí sola no demuestra una relación causal entre las variables.

### 3.2. Ecuación de la regresión lineal simple

La regresión lineal simple supone que la relación entre una variable predictora ($X$) y una variable respuesta ($Y$) puede representarse mediante una línea recta.

La ecuación del modelo es:

$$
Y_i = \beta_0 + \beta_1 X_i + \varepsilon_i
$$

Donde:

- $Y_i$: valor observado de la variable respuesta.
- $X_i$: valor observado de la variable predictora.
- $\beta_0$: **intercepto**, valor esperado de $Y$ cuando $X=0$.
- $\beta_1$: **pendiente**, cambio esperado en $Y$ por cada unidad de aumento en $X$.
- $\varepsilon_i$: **término de error**, que representa la variabilidad de $Y$ que no es explicada por la relación lineal.

Los coeficientes $\beta_0$ y $\beta_1$ son desconocidos y deben estimarse utilizando los datos disponibles.

Una vez ajustado el modelo, obtenemos la ecuación de regresión:

$$
\hat{Y}_i = b_0 + b_1 X_i
$$

Donde:

- $\hat{Y}_i$: valor estimado por el modelo.
- $b_0$: intercepto estimado.
- $b_1$: pendiente estimada.

La diferencia entre el valor observado y el estimado se denomina **residuo**:

$$
e_i = Y_i - \hat{Y}_i
$$

Los residuos permiten evaluar qué tan bien representa el modelo las observaciones.

### Ejemplo: temperatura y humedad relativa

Supongamos que ajustamos un modelo para estimar la humedad relativa a partir de la temperatura del aire y obtenemos la siguiente ecuación:

$$
\widehat{Humedad} = 110 - 2.5 \times Temperatura
$$

**¿Cómo interpretamos los coeficientes?**

**Intercepto ($b_0=110$):**

El modelo estima una humedad relativa de 110 % cuando la temperatura es 0 °C.

Este resultado no es físicamente posible, ya que la humedad relativa no puede superar el 100 %. Esto ilustra que una ecuación de regresión puede producir estimaciones fuera de los límites físicos de una variable.

**Pendiente ($b_1=-2.5$):**

Por cada aumento de 1 °C en la temperatura, el modelo estima una disminución promedio de 2.5 puntos porcentuales en la humedad relativa.

El signo negativo indica una relación inversa entre ambas variables.

**Estimación para una temperatura de 20 °C:**

$$
\widehat{Humedad} = 110 - 2.5(20) = 60\%
$$

Por tanto, el modelo estima una humedad relativa del 60 % cuando la temperatura es 20 °C.

**Importante:** los coeficientes de este ejemplo son ilustrativos. En las siguientes secciones calcularemos los coeficientes reales utilizando las observaciones de MeteoChillán.

### Interpretación física

Aunque la temperatura y la humedad relativa presentan una relación negativa, esta no es necesariamente lineal.

La humedad relativa depende de la presión de vapor de agua y de la presión de vapor de saturación, que aumenta de forma no lineal con la temperatura.

Por ello, la regresión lineal representa una **aproximación estadística** de la relación observada, no una descripción completa de los procesos físicos involucrados.

### 3.3. Método de mínimos cuadrados

En la sección anterior vimos que una regresión lineal simple se representa mediante la ecuación:

$$
\hat{Y}_i = b_0 + b_1 X_i
$$

Pero ¿cómo determinamos los valores de la pendiente ($b_1$) y del intercepto ($b_0$)?

Para ello utilizamos el **método de mínimos cuadrados ordinarios (Ordinary Least Squares, OLS)**.

#### ¿Cómo funciona?

Imaginemos un diagrama de dispersión donde cada punto representa una observación de temperatura y humedad relativa.

Podemos dibujar muchas rectas diferentes sobre estos puntos. Sin embargo, algunas representarán mejor las observaciones que otras.

El método de mínimos cuadrados busca encontrar la recta que **minimiza la suma de las diferencias al cuadrado entre los valores observados y los estimados**.

Estas diferencias se denominan **residuos**:

$$
e_i = Y_i - \hat{Y}_i
$$

La función que buscamos minimizar es:

$$
SSE = \sum_{i=1}^{n}(Y_i-\hat{Y}_i)^2
$$

Donde:

- $SSE$: suma de los errores al cuadrado (*Sum of Squared Errors*).
- $Y_i$: valor observado.
- $\hat{Y}_i$: valor estimado por el modelo.
- $n$: número de observaciones.

#### ¿Por qué elevamos los residuos al cuadrado?

Principalmente por dos razones:

1. **Evitar que los errores positivos y negativos se cancelen.** Si simplemente sumáramos los residuos, las sobreestimaciones podrían compensar las subestimaciones.
2. **Penalizar los errores grandes.** Al elevar los residuos al cuadrado, los errores de mayor magnitud tienen más influencia en el ajuste.

Por ejemplo:

| Humedad observada (%) | Humedad estimada (%) | Residuo | Residuo² |
|---|---|---|---|
| 80 | 75 | 5 | 25 |
| 60 | 65 | -5 | 25 |
| 90 | 88 | 2 | 4 |

En este ejemplo:

$$
SSE = 25 + 25 + 4 = 54
$$

El método OLS busca los valores de $b_0$ y $b_1$ que produzcan el menor SSE posible.

#### ¿Cómo se calculan los coeficientes?

Para una regresión lineal simple con intercepto, la pendiente se calcula como:

$$
b_1 = \frac{\sum_{i=1}^{n}(X_i-\bar{X})(Y_i-\bar{Y})}
{\sum_{i=1}^{n}(X_i-\bar{X})^2}
$$

Y el intercepto:

$$
b_0 = \bar{Y} - b_1\bar{X}
$$

Donde:

- $\bar{X}$: media de la variable predictora.
- $\bar{Y}$: media de la variable respuesta.

Estas ecuaciones permiten calcular directamente los coeficientes de la recta que minimiza el SSE.

En Python no necesitamos realizar estos cálculos manualmente, ya que existen librerías que implementan el método OLS.

Sin embargo, comprender su funcionamiento es importante para interpretar los resultados y reconocer sus limitaciones.

**Importante:** el método OLS encuentra la recta que minimiza el SSE en los datos utilizados para el ajuste. Esto no garantiza que la relación sea físicamente lineal ni que el modelo produzca buenas estimaciones para observaciones nuevas.

## 4. Exploración de la relación entre temperatura y humedad relativa

Antes de ajustar un modelo de regresión lineal, es importante visualizar la relación entre las variables.

En este caso utilizaremos:

- **Variable predictora (X):** temperatura del aire (°C).
- **Variable respuesta (Y):** humedad relativa (%).

Cada punto del diagrama de dispersión representará una observación horaria de MeteoChillán.

### 4.1. Selección de las variables

Para construir el modelo necesitamos observaciones válidas de ambas variables.

Aunque los datos fueron procesados previamente, existen horas sin información. Por ello, seleccionaremos las columnas de temperatura y humedad relativa y eliminaremos únicamente los registros donde falte alguna de estas variables.

### 4.2. Diagrama de dispersión

El diagrama de dispersión nos permitirá identificar:

1. **Dirección:** si la relación entre las variables es positiva o negativa.
2. **Forma:** si la relación parece aproximadamente lineal o presenta curvaturas.
3. **Dispersión:** cuánto varía la humedad relativa para una misma temperatura.
4. **Valores extremos:** observaciones que se alejan del comportamiento general.

En esta primera visualización no ajustaremos ninguna recta. Primero observaremos la distribución de los datos.

In [ ]:
# Seleccionar las variables para la regresión
df_reg = df_final[["Temperatura", "Humedad"]].dropna().copy()

# Definir las variables
X = df_reg["Temperatura"]
Y = df_reg["Humedad"]

print(f"Observaciones disponibles: {len(df_reg)}")

# Diagrama de dispersión
fig, ax = plt.subplots(figsize=(9, 6))

sns.scatterplot(
    data=df_reg,
    x="Temperatura",
    y="Humedad",
    alpha=0.25,
    s=20,
    color="steelblue",
    ax=ax
)

ax.set(
    title="Relación entre temperatura y humedad relativa — MeteoChillán",
    xlabel="Temperatura (°C)",
    ylabel="Humedad relativa (%)"
)

plt.tight_layout()
plt.show()

### 4.3. Interpretación del diagrama

Observa el gráfico y responde:

1. ¿La relación entre temperatura y humedad relativa es positiva o negativa?
2. ¿Los puntos siguen aproximadamente una línea recta o presentan una relación curva?
3. ¿Existe dispersión en los valores de humedad relativa para una misma temperatura?
4. ¿Consideras que una regresión lineal simple puede representar razonablemente esta relación?

**Recuerda:** una correlación fuerte no significa necesariamente que todos los puntos estén próximos a una recta.

En la siguiente sección ajustaremos una regresión lineal mediante el método de mínimos cuadrados y compararemos la recta estimada con las observaciones.

## 5. Ajuste del modelo de regresión lineal simple

Una vez explorada la relación entre temperatura y humedad relativa, ajustaremos un modelo de regresión lineal utilizando el **método de mínimos cuadrados ordinarios (OLS)**.

Nuestro objetivo es encontrar la recta que mejor represente la relación entre ambas variables.

El modelo se expresa como:

$$
\widehat{Humedad}_i = b_0 + b_1 \cdot Temperatura_i
$$

Donde:

- $b_0$: intercepto de la recta.
- $b_1$: pendiente de la recta.
- $\widehat{Humedad}_i$: humedad relativa estimada por el modelo.

### 5.1. Ajuste mediante Statsmodels

Utilizaremos la librería **Statsmodels**, que permite ajustar modelos estadísticos y obtener información detallada sobre sus coeficientes.

Los pasos son:

1. Definir la temperatura como variable predictora ($X$).
2. Definir la humedad relativa como variable respuesta ($Y$).
3. Incorporar una constante para estimar el intercepto.
4. Ajustar el modelo mediante OLS.
5. Obtener los coeficientes de la recta.

En esta primera etapa ajustaremos el modelo utilizando todas las observaciones válidas, con el objetivo de comprender la relación estadística entre las variables.

Posteriormente, separaremos los datos en conjuntos de entrenamiento y prueba para evaluar su capacidad de estimación sobre observaciones no utilizadas durante el ajuste.

In [ ]:
# Agregar una constante para estimar el intercepto
X_ols = sm.add_constant(X)

# Definir y ajustar el modelo
modelo_ols = sm.OLS(Y, X_ols).fit()

# Obtener los coeficientes
b0 = modelo_ols.params["const"]
b1 = modelo_ols.params["Temperatura"]

# Mostrar resultados
print("Coeficientes del modelo de regresión lineal")

print(f"Intercepto (b0): {b0:.3f}")
print(f"Pendiente (b1): {b1:.3f}")

print("\nEcuación de regresión:")
print(f"Humedad estimada = {b0:.3f} + ({b1:.3f} × Temperatura)")

### 5.2. Interpretación de los coeficientes

Los coeficientes obtenidos permiten interpretar cuantitativamente la relación entre temperatura y humedad relativa.

**Intercepto ($b_0$)**

Representa la humedad relativa estimada cuando la temperatura es 0 °C.

Aunque tiene una interpretación matemática, debemos comprobar si su valor es físicamente razonable.

**Pendiente ($b_1$)**

Representa el cambio promedio estimado en la humedad relativa por cada aumento de 1 °C en la temperatura.

- Si $b_1 > 0$, la relación es positiva.
- Si $b_1 < 0$, la relación es negativa.
- Si $b_1 = 0$, la recta no presenta cambios en la humedad relativa asociados con la temperatura.

Por ejemplo, si obtenemos una pendiente de $-2.5$, significa que por cada aumento de 1 °C, el modelo estima una disminución promedio de **2.5 puntos porcentuales** en la humedad relativa.

Esta interpretación describe una asociación estadística y no demuestra, por sí sola, una relación causal.

### Actividad

Utilizando los coeficientes obtenidos:

1. Interpreta el significado de la pendiente y sus unidades.
2. ¿Qué humedad relativa estima el modelo cuando la temperatura es 15 °C?
3. ¿El intercepto tiene una interpretación física razonable?

In [ ]:
# Obtener las estimaciones del modelo
Y_estimado = modelo_ols.predict(X_ols)

# Crear gráfico
fig, ax = plt.subplots(figsize=(9, 6))

# Observaciones
sns.scatterplot(
    x=X,
    y=Y,
    alpha=0.20,
    s=20,
    color="steelblue",
    label="Observaciones",
    ax=ax
)

# Recta de regresión
# Dos temperaturas: mínima y máxima observada

x_recta = np.array([X.min(), X.max()])

# Calcular la humedad estimada en ambos extremos

y_recta = b0 + b1 * x_recta

# Dibujar la recta de regresión

ax.plot(

    x_recta,

    y_recta,

    color="red",

    linewidth=2.5,

    label="Regresión lineal"

)

# Mostrar ecuación en el gráfico
ecuacion = f"Humedad = {b0:.2f} {b1:+.2f} × Temperatura"

ax.text(
    0.05, 0.05,
    ecuacion,
    transform=ax.transAxes,
    fontsize=11,
    bbox=dict(facecolor="white", alpha=0.9, edgecolor="gray")
)

ax.set(
    title="Regresión lineal entre temperatura y humedad relativa — MeteoChillán",
    xlabel="Temperatura (°C)",
    ylabel="Humedad relativa (%)"
)

ax.legend()
plt.tight_layout()
plt.show()

## 6. Evaluación del modelo de regresión lineal

Hasta ahora hemos ajustado una regresión lineal para estimar la humedad relativa a partir de la temperatura del aire.

Sin embargo, obtener una ecuación no significa necesariamente que el modelo represente adecuadamente las observaciones.

Para evaluar su desempeño debemos comparar los valores observados ($Y_i$) con los valores estimados ($\hat{Y}_i$).

Utilizaremos tres métricas:

- **Coeficiente de determinación ($R^2$):** indica qué proporción de la variabilidad observada es explicada por el modelo.
- **Error absoluto medio (MAE):** cuantifica la magnitud promedio de los errores.
- **Raíz del error cuadrático medio (RMSE):** cuantifica los errores, dando mayor peso a los de gran magnitud.

### 6.1. Coeficiente de determinación ($R^2$)

El coeficiente de determinación se calcula como:

$$
R^2 = 1 - \frac{\sum_{i=1}^{n}(Y_i-\hat{Y}_i)^2}
{\sum_{i=1}^{n}(Y_i-\bar{Y})^2}
$$

Donde:

- $Y_i$: valor observado.
- $\hat{Y}_i$: valor estimado.
- $\bar{Y}$: media de los valores observados.

En una regresión lineal ajustada por OLS con intercepto y evaluada sobre los mismos datos, $R^2$ toma valores entre 0 y 1.

Por ejemplo:

- $R^2=0$: el modelo no mejora la estimación constante basada en la media observada.
- $R^2=0.50$: el modelo explica el 50 % de la variabilidad observada.
- $R^2=0.90$: el modelo explica el 90 % de la variabilidad observada.

**Relación con la correlación**

En una regresión lineal simple con intercepto, cuando el modelo se evalúa sobre los mismos datos utilizados para ajustarlo:

$$
R^2 = r^2
$$

Donde $r$ es el coeficiente de correlación de Pearson.

En la actividad anterior encontramos una correlación de aproximadamente $r=-0.87$ entre temperatura y humedad relativa.

Por tanto, esperamos obtener:

$$
R^2 \approx (-0.87)^2 \approx 0.76
$$

Esto significa que aproximadamente el 76 % de la variabilidad observada en la humedad relativa puede ser representada por la relación lineal con la temperatura.

**Importante:** un valor elevado de $R^2$ no garantiza que el modelo sea físicamente adecuado ni que funcione bien con observaciones nuevas.

### 6.2. Error absoluto medio (MAE)

El MAE representa el promedio de las diferencias absolutas entre los valores observados y estimados.

$$
MAE = \frac{1}{n}\sum_{i=1}^{n}|Y_i-\hat{Y}_i|
$$

Por ejemplo, un MAE de 8 puntos porcentuales significa que las estimaciones de humedad relativa presentan un error absoluto promedio de 8 puntos porcentuales.

**Interpretación:**

- MAE = 0: estimaciones perfectas.
- Cuanto menor sea el MAE, menor será el error promedio.
- Tiene las mismas unidades que la variable respuesta.

### 6.3. Raíz del error cuadrático medio (RMSE)

El RMSE se calcula como:

$$
RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(Y_i-\hat{Y}_i)^2}
$$

A diferencia del MAE, el RMSE penaliza más los errores grandes porque primero eleva las diferencias al cuadrado.

**Interpretación:**

- RMSE = 0: estimaciones perfectas.
- Cuanto menor sea el RMSE, mejor será el ajuste.
- Tiene las mismas unidades que la variable respuesta.

### ¿Cuál es la diferencia entre MAE y RMSE?

El MAE describe la magnitud promedio de los errores absolutos.

El RMSE es más sensible a los errores grandes.

Por ejemplo, si un modelo estima correctamente la mayoría de las observaciones, pero presenta errores muy grandes en algunos casos, el RMSE aumentará más que el MAE.

En nuestro caso, ambas métricas se expresarán en **puntos porcentuales de humedad relativa**.

**Nota:** inicialmente calcularemos estas métricas utilizando los datos empleados para ajustar el modelo. Estos resultados describen el ajuste dentro de la muestra y no constituyen una evaluación independiente de su capacidad predictiva.

In [ ]:
# Valores observados y estimados
y_obs = Y
y_est = Y_estimado

# Calcular métricas
r2 = r2_score(y_obs, y_est)
mae = mean_absolute_error(y_obs, y_est)
rmse = np.sqrt(mean_squared_error(y_obs, y_est))

# Correlación de Pearson
r, _ = stats.pearsonr(X, Y)

# Mostrar resultados
print("Evaluación del modelo de regresión lineal")

print(f"R²:   {r2:.3f}")
print(f"MAE:  {mae:.2f} puntos porcentuales")
print(f"RMSE: {rmse:.2f} puntos porcentuales")

print("\nRelación entre correlación y regresión:")
print(f"Pearson (r): {r:.3f}")
print(f"r²:          {r**2:.3f}")
print(f"R²:          {r2:.3f}")

### 6.4. Interpretación del desempeño

A partir de los resultados obtenidos, responde:

1. **Coeficiente de determinación:** ¿qué porcentaje de la variabilidad observada en la humedad relativa es explicado por el modelo?

2. **Relación con la correlación:** ¿coincide el valor de $R^2$ con el cuadrado del coeficiente de Pearson obtenido anteriormente?

3. **Error absoluto medio:** ¿cuál es la magnitud promedio de los errores en las estimaciones de humedad relativa?

4. **RMSE:** ¿es mayor o menor que el MAE? ¿Por qué ocurre esto?

5. **Interpretación general:** ¿consideras que el modelo representa adecuadamente la relación entre temperatura y humedad relativa?

### Una consideración importante

Aunque un modelo presente un $R^2$ relativamente elevado, todavía puede tener limitaciones importantes.

Por ejemplo, una regresión lineal puede:

- Sobreestimar o subestimar sistemáticamente la humedad relativa en determinados rangos de temperatura.
- Producir valores estimados fuera del intervalo físico de 0 a 100 %.
- Presentar errores que cambian según la temperatura.
- No representar adecuadamente relaciones no lineales.

Para identificar estos problemas necesitamos estudiar los **residuos del modelo**.

## 7. Análisis de residuos

Hasta ahora hemos evaluado el modelo mediante $R^2$, MAE y RMSE. Sin embargo, estas métricas resumen el desempeño en un único valor y no permiten identificar todos los problemas del ajuste.

Para comprender mejor el comportamiento del modelo necesitamos analizar sus **residuos**.

### 7.1. ¿Qué son los residuos?

Un residuo es la diferencia entre el valor observado y el valor estimado por el modelo:

$$
e_i = Y_i - \hat{Y}_i
$$

Donde:

- $e_i$: residuo de la observación $i$.
- $Y_i$: humedad relativa observada.
- $\hat{Y}_i$: humedad relativa estimada.

**Interpretación:**

- **Residuo positivo ($e_i>0$):** el modelo subestima la humedad relativa.
- **Residuo negativo ($e_i<0$):** el modelo sobreestima la humedad relativa.
- **Residuo igual a cero ($e_i=0$):** la estimación coincide con la observación.

Por ejemplo:

| Humedad observada (%) | Humedad estimada (%) | Residuo | Interpretación |
|---|---|---|---|
| 80 | 75 | +5 | Subestimación |
| 60 | 65 | -5 | Sobreestimación |
| 90 | 90 | 0 | Estimación exacta |

### 7.2. ¿Por qué analizamos los residuos?

Un modelo de regresión lineal puede presentar un $R^2$ elevado y, aun así, tener problemas importantes.

El análisis de residuos permite investigar:

1. **Linealidad:** si una recta representa adecuadamente la relación entre las variables.
2. **Varianza constante (homocedasticidad):** si la dispersión de los errores es aproximadamente constante.
3. **Independencia:** si los errores presentan dependencia temporal.
4. **Distribución de los errores:** si los residuos presentan una distribución aproximadamente normal.

Estos aspectos son importantes para evaluar la adecuación del modelo y la validez de determinadas pruebas estadísticas.

**Importante:** la normalidad de los residuos no es un requisito para calcular los coeficientes mediante mínimos cuadrados, pero sí es relevante para algunas pruebas de hipótesis e intervalos de confianza convencionales.

### 7.3. ¿Cómo debería verse un gráfico de residuos?

En un modelo lineal adecuado, esperamos que:

- Los residuos se distribuyan alrededor de cero.
- No aparezcan patrones sistemáticos o curvaturas.
- La dispersión sea relativamente constante.
- No existan estructuras temporales importantes.

Si observamos patrones definidos, puede ser una señal de que el modelo no representa completamente la relación entre las variables.

En nuestro caso, debemos recordar que trabajamos con **observaciones meteorológicas horarias**, por lo que es probable que los residuos presenten dependencia temporal.

In [ ]:
# Calcular los residuos
residuos = y_obs - y_est

# Crear un DataFrame para analizar los resultados
df_residuos = pd.DataFrame({
    "Temperatura": X,
    "Humedad_observada": y_obs,
    "Humedad_estimada": y_est,
    "Residuo": residuos
})

# Mostrar las primeras observaciones
display(df_residuos.head())

# Estadísticos descriptivos de los residuos
print("Estadísticos de los residuos:")
display(df_residuos["Residuo"].describe().round(2))

print(f"\nMedia de los residuos: {residuos.mean():.6f}")

### 7.4. Visualización de los residuos

Construiremos tres gráficos para analizar el comportamiento de los errores.

**A. Residuos frente a temperatura**

Permite identificar relaciones no lineales y cambios en la dispersión de los errores.

Si el modelo lineal es adecuado, no deberíamos observar patrones sistemáticos.

**B. Distribución de los residuos**

Utilizaremos un histograma para examinar la forma de la distribución.

Observaremos si los errores se concentran alrededor de cero, si presentan asimetría o si existen valores extremos.

**C. Residuos a lo largo del tiempo**

Permite identificar periodos con errores sistemáticos y posibles patrones temporales.

En series meteorológicas, las observaciones consecutivas suelen estar relacionadas. Por ello, los residuos pueden presentar autocorrelación.

La presencia de autocorrelación puede afectar la validez de las pruebas estadísticas convencionales, aunque no impide calcular los coeficientes de regresión.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# A. Residuos frente a temperatura
sns.scatterplot(
    data=df_residuos,
    x="Temperatura",
    y="Residuo",
    alpha=0.25,
    s=15,
    ax=axes[0]
)

axes[0].axhline(0, color="red", linestyle="--")
axes[0].set(
    title="A. Residuos vs. temperatura",
    xlabel="Temperatura (°C)",
    ylabel="Residuo (puntos porcentuales)"
)

# B. Distribución de residuos
sns.histplot(
    data=df_residuos,
    x="Residuo",
    bins=40,
    kde=True,
    ax=axes[1]
)

axes[1].axvline(0, color="red", linestyle="--")
axes[1].set(
    title="B. Distribución de residuos",
    xlabel="Residuo (puntos porcentuales)",
    ylabel="Frecuencia"
)

# C. Residuos a lo largo del tiempo
axes[2].plot(
    df_residuos.index,
    df_residuos["Residuo"],
    linewidth=0.6,
    alpha=0.7
)

axes[2].axhline(0, color="red", linestyle="--")
axes[2].set(
    title="C. Evolución temporal de los residuos",
    xlabel="Fecha",
    ylabel="Residuo (puntos porcentuales)"
)

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 7.5. Interpretación de los resultados

Observa los tres gráficos y responde:

**A. Residuos frente a temperatura**

1. ¿Los residuos se distribuyen aleatoriamente alrededor de cero?
2. ¿Observas algún patrón o curvatura?
3. ¿La dispersión de los residuos es similar para todas las temperaturas?

**B. Distribución de residuos**

4. ¿Los residuos se concentran alrededor de cero?
5. ¿La distribución es aproximadamente simétrica?
6. ¿Existen residuos de gran magnitud?

**C. Evolución temporal**

7. ¿Se observan periodos consecutivos con residuos positivos o negativos?
8. ¿Qué podría indicar este comportamiento sobre la independencia de los errores?

### Reflexión

La regresión lineal permite representar una parte importante de la relación entre temperatura y humedad relativa. Sin embargo, ambas variables están relacionadas mediante procesos atmosféricos que no son estrictamente lineales.

Además, la humedad relativa depende del contenido de vapor de agua, que no está incluido como predictor en nuestro modelo.

Por tanto, aunque obtengamos un buen ajuste estadístico, es importante reconocer que el modelo constituye una simplificación del comportamiento físico del sistema.

**Un buen desempeño estadístico no garantiza que todos los supuestos del modelo se cumplan.**

## 8. Significancia estadística e intervalos de confianza

Hasta ahora hemos ajustado una regresión lineal y evaluado su desempeño mediante $R^2$, MAE, RMSE y el análisis de residuos.

Sin embargo, todavía podemos plantearnos una pregunta:

**¿La relación lineal observada entre temperatura y humedad relativa es estadísticamente significativa?**

Para responderla utilizaremos pruebas de hipótesis e intervalos de confianza asociados a los coeficientes de regresión.

### 8.1. Prueba de hipótesis para la pendiente

Recordemos la ecuación del modelo:

$$
\hat{Y}_i = b_0 + b_1 X_i
$$

La pendiente ($b_1$) representa el cambio estimado en la humedad relativa por cada aumento de 1 °C en la temperatura.

Queremos determinar si existe evidencia estadística de que la pendiente poblacional ($\beta_1$) es diferente de cero.

Para ello formulamos dos hipótesis:

**Hipótesis nula ($H_0$):**

$$
H_0: \beta_1 = 0
$$

No existe una relación lineal entre temperatura y humedad relativa en la población estudiada.

**Hipótesis alternativa ($H_1$):**

$$
H_1: \beta_1 \neq 0
$$

Existe una relación lineal entre ambas variables.

### 8.2. Estadístico t y valor p

La prueba utiliza el estadístico:

$$
t = \frac{b_1}{SE(b_1)}
$$

Donde:

- $b_1$: pendiente estimada.
- $SE(b_1)$: error estándar de la pendiente.

El error estándar representa la incertidumbre asociada a la estimación del coeficiente.

A partir del estadístico $t$, se calcula el **valor p**.

Recordemos que el valor p representa la probabilidad, suponiendo que la hipótesis nula es verdadera y que se cumplen los supuestos de la prueba, de obtener un estadístico tan extremo como el observado o más.

Utilizaremos un nivel de significancia:

$$
\alpha = 0.05
$$

La interpretación convencional es:

- **Si $p < 0.05$:** rechazamos $H_0$. Existe evidencia estadística de una relación lineal.
- **Si $p \geq 0.05$:** no rechazamos $H_0$. No tenemos evidencia suficiente para afirmar que la pendiente es diferente de cero.

**Importante:** un valor p pequeño no significa que el modelo sea preciso, que la relación sea causal o que la pendiente tenga una magnitud importante.

### 8.3. Intervalos de confianza

Además de estimar los coeficientes, podemos calcular sus intervalos de confianza.

Un intervalo de confianza del 95 % para la pendiente tiene la forma:

$$
IC_{95\%}(\beta_1) =
b_1 \pm t_{0.975,n-2}\,SE(b_1)
$$

El intervalo proporciona un rango de valores compatibles con los datos y los supuestos estadísticos utilizados.

Por ejemplo, supongamos que obtenemos:

$$
b_1 = -2.5
$$

Con un intervalo de confianza del 95 %:

$$
IC_{95\%} = [-2.7,\,-2.3]
$$

Esto indica que los valores de la pendiente comprendidos entre -2.7 y -2.3 puntos porcentuales por °C son compatibles con los datos, bajo los supuestos del modelo.

Si el intervalo no incluye cero, la pendiente es estadísticamente significativa al nivel del 5 % en una prueba bilateral equivalente.

### 8.4. Una limitación importante: autocorrelación temporal

Nuestros datos meteorológicos tienen resolución horaria.

Las observaciones consecutivas no son completamente independientes. Por ejemplo, la temperatura y la humedad relativa de las 14:00 suelen estar relacionadas con sus valores de las 13:00.

Esta dependencia temporal también puede aparecer en los residuos del modelo.

Las pruebas t y los intervalos de confianza convencionales de OLS suponen, entre otras condiciones, que los errores no presentan autocorrelación.

Cuando este supuesto no se cumple:

- Los errores estándar convencionales pueden ser incorrectos.
- Los intervalos de confianza pueden resultar demasiado estrechos.
- Los valores p pueden sugerir una significancia estadística mayor de la que realmente existe.

Por ello, primero examinaremos los resultados convencionales de OLS y después utilizaremos una corrección que considera la autocorrelación temporal.

In [ ]:
# Mostrar el resumen estadístico completo
print(modelo_ols.summary())

In [ ]:
# Extraer los coeficientes y sus resultados estadísticos
resultados = pd.DataFrame({
    "Coeficiente": modelo_ols.params,
    "Error estándar": modelo_ols.bse,
    "Estadístico t": modelo_ols.tvalues,
    "Valor p": modelo_ols.pvalues,
    "IC 95% inferior": modelo_ols.conf_int()[0],
    "IC 95% superior": modelo_ols.conf_int()[1]
})

display(resultados.round(4))


La tabla de resultados  proporciona información sobre los coeficientes estimados, la calidad del ajuste y las pruebas estadísticas del modelo.




#### Prueba t y valor p de la pendiente

**Valores obtenidos:**

- Estadístico t: **-139.073**
- Valor p: **< 0.001**

La **prueba t** permite evaluar si la pendiente es estadísticamente diferente de cero.

Las hipótesis son:

$$
H_0: \beta_1 = 0
$$

$$
H_1: \beta_1 \neq 0
$$

Como el valor p es menor que 0.05, rechazamos la hipótesis nula según la prueba convencional.

**Interpretación:** existe evidencia estadística de una relación lineal negativa entre temperatura y humedad relativa, bajo los supuestos de la prueba.

Esto no demuestra causalidad ni garantiza que el modelo produzca buenas estimaciones.

#### Intervalo de confianza del 95 %

**Valores obtenidos para la pendiente:**

$$
IC_{95\%} = [-2.509,\,-2.439]
$$

El intervalo de confianza representa un rango de valores compatibles con la pendiente poblacional, considerando la incertidumbre de su estimación y los supuestos estadísticos.

**Interpretación:** el intervalo es completamente negativo y no incluye cero, lo que coincide con el resultado de la prueba t.

Además, el intervalo es relativamente estrecho, lo que indica una estimación aparentemente precisa bajo los supuestos convencionales.

#### Prueba de Durbin-Watson

**Valor obtenido:**

$$
DW = 0.057
$$

El estadístico de **Durbin-Watson** permite identificar indicios de autocorrelación de primer orden en los residuos.

Su interpretación aproximada es:

- **DW cercano a 2:** ausencia de autocorrelación de primer orden.
- **DW cercano a 0:** autocorrelación positiva fuerte.
- **DW cercano a 4:** autocorrelación negativa fuerte.

**Interpretación:** nuestro valor de 0.057 indica una autocorrelación positiva muy fuerte entre residuos consecutivos.

Esto coincide con los patrones observados en el gráfico temporal de residuos.

La autocorrelación es importante porque puede afectar los errores estándar, los valores p y los intervalos de confianza convencionales.

#### Pruebas de normalidad de los residuos

La tabla también presenta dos pruebas estadísticas:

| Prueba | Valor p | Resultado |
|---|---|---|
| Omnibus | < 0.001 | Se rechaza la hipótesis de normalidad |
| Jarque-Bera | < 0.001 | Se rechaza la hipótesis de normalidad |

Ambas pruebas evalúan si la distribución de los residuos es compatible con una distribución normal.

**Interpretación:** los resultados indican desviaciones estadísticamente significativas respecto de la normalidad.

Sin embargo, como tenemos 6.398 observaciones, incluso desviaciones pequeñas pueden producir valores p muy bajos. Por ello, estos resultados deben interpretarse junto con el histograma de residuos.



### En conclusión:


La regresión lineal permite representar una parte importante de la relación entre temperatura y humedad relativa. Sin embargo, los resultados también muestran que un buen ajuste y una pendiente estadísticamente significativa no garantizan que se cumplan todos los supuestos del modelo.

En particular, la fuerte autocorrelación temporal indica que los errores estándar y los intervalos de confianza convencionales deben interpretarse con precaución.

En la siguiente sección utilizaremos errores estándar robustos a la autocorrelación para comprobar cómo cambia la incertidumbre asociada a los coeficientes.

### 8.5. Errores estándar robustos a la autocorrelación

Para considerar la dependencia temporal de los residuos utilizaremos el método **Newey-West**, también conocido como estimador HAC (*Heteroskedasticity and Autocorrelation Consistent*).

Este método permite calcular errores estándar robustos frente a determinadas formas de heterocedasticidad y autocorrelación.

La principal diferencia es que:

- Los **coeficientes estimados** mediante OLS no cambian.
- Los **errores estándar** pueden cambiar.
- Los **intervalos de confianza y valores p** pueden cambiar.

En nuestro ejemplo utilizaremos un máximo de 24 rezagos horarios para considerar la dependencia de los errores durante aproximadamente un día.

Este valor es una elección ilustrativa, no una garantía de que toda la dependencia temporal quede representada.

El objetivo es mostrar cómo las características temporales de los datos pueden influir en la interpretación estadística de una regresión.

In [ ]:
# Calcular resultados con errores estándar robustos HAC
modelo_hac = modelo_ols.get_robustcov_results(
    cov_type="HAC",
    maxlags=24
)

# Construir tabla comparativa
comparacion = pd.DataFrame({
    "Coeficiente": modelo_ols.params.values,
    "EE convencional": modelo_ols.bse.values,
    "EE HAC": modelo_hac.bse,
    "Valor p convencional": modelo_ols.pvalues.values,
    "Valor p HAC": modelo_hac.pvalues,
    "IC HAC inferior": modelo_hac.conf_int()[:, 0],
    "IC HAC superior": modelo_hac.conf_int()[:, 1]
}, index=modelo_ols.params.index)

display(comparacion.round(4))


### 8.6. Interpretación de los resultados: OLS convencional y HAC

En la sección anterior identificamos una fuerte autocorrelación temporal en los residuos de nuestra regresión (Durbin-Watson = 0.057).

Esto significa que los errores de observaciones consecutivas están relacionados, lo que puede afectar la estimación convencional de la incertidumbre de los coeficientes.

Para investigar este efecto aplicamos la corrección **HAC (Newey-West)**, que permite calcular errores estándar robustos frente a determinadas formas de heterocedasticidad y autocorrelación.

### Comparación de resultados

| Indicador | OLS convencional | OLS con HAC |
|---|---:|---:|
| Pendiente | -2.4737 | -2.4737 |
| Error estándar | 0.0178 | 0.0559 |
| Valor p | < 0.001 | < 0.001 |
| IC 95 % | [-2.5086, -2.4388] | [-2.5832, -2.3642] |

**¿Qué observamos?**

**1. La pendiente no cambia**

En ambos casos obtenemos una pendiente de -2.4737.

Esto significa que, por cada aumento de 1 °C, la humedad relativa estimada disminuye aproximadamente 2.47 puntos porcentuales.

HAC no modifica los coeficientes de regresión, sino la incertidumbre asociada a sus estimaciones.

**2. El error estándar aumenta**

El error estándar pasa de 0.0178 a 0.0559, aumentando aproximadamente 3.1 veces.

Esto indica que la estimación convencional sugería una precisión mayor que la obtenida al considerar la dependencia temporal.

**3. El intervalo de confianza se amplía**

El intervalo convencional es:

$$
IC_{95\%}=[-2.5086,-2.4388]
$$

Mientras que el intervalo HAC es:

$$
IC_{95\%}=[-2.5832,-2.3642]
$$

El intervalo HAC es más amplio porque considera una mayor incertidumbre en la estimación de la pendiente.

**4. La pendiente continúa siendo estadísticamente significativa**

En ambos casos, el valor p es menor que 0.001.

Por tanto, aunque la incertidumbre aumenta, seguimos encontrando evidencia estadística de una asociación lineal negativa entre temperatura y humedad relativa.

### Conclusión

**La corrección HAC no cambia la relación estimada entre las variables, pero sí cambia nuestra confianza en la precisión de los coeficientes.**

Este ejemplo demuestra que utilizar muchas observaciones no garantiza que todas aporten información estadística independiente.

En las series meteorológicas, donde existe dependencia temporal, es importante considerar la autocorrelación al interpretar los resultados de una regresión.


## 9. Conclusiones

En esta actividad aprendimos a construir e interpretar un modelo de regresión lineal simple utilizando datos meteorológicos de MeteoChillán.

A partir de la relación entre temperatura y humedad relativa:

- Identificamos las diferencias entre correlación y regresión lineal.
- Comprendimos cómo funciona el método de mínimos cuadrados.
- Estimamos e interpretamos los coeficientes de una regresión lineal.
- Evaluamos el ajuste mediante R², MAE y RMSE.
- Analizamos los residuos para identificar posibles limitaciones del modelo.
- Interpretamos la significancia estadística y los intervalos de confianza de los coeficientes.

### ¿Qué aprendimos?

La regresión lineal es una herramienta sencilla e interpretable para representar relaciones entre variables cuantitativas.

Sin embargo, presenta limitaciones cuando las relaciones son no lineales o cuando intervienen procesos que no están representados por las variables predictoras.

Además, un buen ajuste a los datos utilizados para construir el modelo no garantiza que este funcione adecuadamente con nuevas observaciones.

